In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import trimesh
import fvdb
import argparse
from pprint import pprint
import tqdm
import h5py
from skimage import measure
from meshplot import plot
from scipy.spatial.distance import cdist
import joblib
import numpy as np
from scipy.spatial import KDTree
import igl
import fvdb
import matplotlib.pyplot as plt
import fvdb.nn as fvnn
import torch


# Import ssu packages
sys.path.append('../src')
sys.path.append('../config')
# config packages
import read_config
# src packages
import eval
import utils
import models
from logger import wandb_logging
from data_loader import ABC_dataset_loader
from utils import fvdb_utils as fu
from utils import ssu_tools as st 
from utils import mesh_tools as mt

In [21]:
# Imports
import os
import sys
sys.path.append('../src/utils')
sys.path.append('../src/data_utils')
import numpy as np
from torch.utils.data import Dataset
from tqdm import tqdm
import h5py
import joblib
import torch
import fvdb.nn as fvnn
import fvdb
import mesh_tools as mt
import fvdb_utils as fu
import time
sys.path.append('../config')
sys.path.append('../src')
import read_config
from pprint import pprint
from logger import wandb_logging
from utils import ssu_tools as st 
from data_loader import ABC_dataset_loader



In [22]:
config_file = 'config_43_30072025_1100.yaml'

In [23]:
config = read_config.read_yaml_config(f'{config_file}')
    
print("Configuration loaded:")
for key, value in config.items():
    pprint(f"{key}: {value}")

# initialize logging
logger = wandb_logging.WandbLogger(
                    logging=config['logging'],
                    project_name=config['wandb']['project_name'],
                    entity=config['wandb']['entity'],
                    name=config['wandb']['name'],
                    group=config['wandb']['group'],
                    tags=config['wandb']['tags'],
                    notes=config['wandb']['notes'],
                    config=config['wandb']['config'],
                    # resume="allow",
                    # id = '1e7bv85c'
                )
logger.update_config('config_file_name', config_file)

# set reproducibility
st.set_reproducibility(is_reproducible=config['reproducibility']['is_reproducible'],
                        seed=config['reproducibility']['seed'])

# load data
input_dir = config['data']['input_dir']
names_set = os.listdir('/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large')
(train_dataloader, 
val_dataloader, 
test_dataloader) = ABC_dataset_loader.ABCDataLoader(
                                    input_dir=input_dir,
                                    config=config,
                                    # n_samples=20
                                ).get(names_set=names_set)

Configuration loaded:
'logging: True'
"reproducibility: {'is_reproducible': True, 'seed': 42}"
("data: {'input_dir': "
 "'/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large', "
 "'data_description': 'Generated SDF of 32, 64, 128, 256 data using NMC "
 "logic', 'data_name': 'V1_Manual_Extracted_NMC_Data_large', 'data_split': "
 "{'train': 0.6, 'val': 0.2, 'test': 0.2}, 'dataset_grids': [32, 64, 128], "
 "'mask_threshold': 65, 'is_crop': {'train': True, 'val': False, 'test': "
 "False}, 'crops_size': [8, 16, 32], 'crops_size_probability': [0.0, 1.0, "
 "0.0], 'crops_threshold': {8: 25, 16: 200, 32: 1600}, 'n_jobs': -1, "
 "'batch_size': 16, 'shuffle': {'train': True, 'val': False, 'test': False}, "
 "'num_workers': 0}")
("training: {'save_model_dir': "
 "'/data/workspaces/spanwar/results/ssu/save_models', 'model_name': "
 "'42_rerun_40_CNN_vanilla_65_mask_threshold_l1_sign_loss', 'lr': 0.001, "
 "'epochs': 40, 'loss_function': 'l1_sign

  0%|          | 0/3210 [00:00<?, ?it/s]

100%|██████████| 1071/1071 [00:16<00:00, 63.54it/s] 


In [41]:
import torch
import torch.nn as nn
import fvdb.nn as fvnn

class CNN_vanilla(nn.Module):
    def __init__(self, in_channels=3, features=32, out_channels=1, dropout=0.05):
        super(CNN_vanilla, self).__init__()
        
        self.activation = fvnn.SiLU(inplace=True)
        self.encoder = nn.Sequential(
            fvnn.SparseConv3d(in_channels, features, kernel_size=3, stride=1),
            fvnn.Dropout(dropout),
            self.activation,
            fvnn.SparseConv3d(features, features, kernel_size=3, stride=1),
            fvnn.Dropout(dropout),
            self.activation,
            fvnn.SparseConv3d(features, features, kernel_size=3, stride=1),
            fvnn.Dropout(dropout),
            self.activation,
            fvnn.SparseConv3d(features, features, kernel_size=3, stride=1),
            fvnn.Dropout(dropout),
            self.activation,

            fvnn.SparseConv3d(features, features, kernel_size=3, stride=1),
            fvnn.Dropout(dropout),
            self.activation
        )
        
        self.decoder = nn.Sequential(
            fvnn.Dropout(dropout),
            self.activation,
            fvnn.SparseConv3d(features, features, kernel_size=1, stride=1),
            fvnn.Dropout(dropout),
            self.activation,
            fvnn.SparseConv3d(features, features, kernel_size=1, stride=1),
            fvnn.Dropout(dropout),
            self.activation,
            fvnn.SparseConv3d(features, out_channels, kernel_size=1, stride=1)
        )

        self.t_conv = fvnn.SparseConv3d(
            features, features, kernel_size=3, stride=2, transposed=True) #TODO check that this is correct
        self.t_conv2 = fvnn.SparseConv3d(
            features, features, kernel_size=3, stride=2, transposed=True)

    def forward(self, x, out_grid1, out_grid2):
        enc = self.encoder(x)
        x = self.t_conv(enc, out_grid=out_grid1)
        x = self.t_conv2(x, out_grid=out_grid2)
        return self.decoder(x)

In [65]:
model = CNN_vanilla(
    in_channels=13,
    out_channels=config['training']['out_channels'])
trainable_params = st.print_model_summary(model)
logger.update_config('model_parameters', trainable_params)

## optimizer
optimizer = torch.optim.Adam(model.parameters(), 
                            lr=config['training']['lr'], 
                            weight_decay=1e-5)

Total parameters: 179489
Trainable parameters: 179489


In [66]:
import os
import sys
import fvdb
import torch
import wandb
from tqdm import tqdm
from training.loss import LossFunctions

# sys.path.append('../src/utils')
# from ssu_tools import positional_encoding


In [67]:
def positional_encoding(small_vdb, dim):
    '''helps the learning'''
    feat = small_vdb.grid.grid_to_world(small_vdb.grid.ijk.float()).jdata
    feat_x = feat[:, 0:1]  # Extract x-coordinates
    feat_y = feat[:, 1:2]  # Extract y-coordinates
    feat_z = feat[:, 2:3]  # Extract z-coordinates
    half_dim = dim // 2
    emb = torch.arange(
        start=0, end=half_dim, dtype=torch.float32, device=feat.device)
    emb = 2**emb * torch.pi
    emb_x = feat_x.float() * emb[None, :]
    emb_y = feat_y.float() * emb[None, :]
    emb_z = feat_z.float() * emb[None, :]
    new_feat = torch.cat([small_vdb.jdata, emb_x.sin(), emb_x.cos(), emb_y.sin(), emb_y.cos(), emb_z.sin(), emb_z.cos()], dim=-1)
    return fvnn.VDBTensor(small_vdb.grid, small_vdb.grid.jagged_like(new_feat))

In [68]:
class ModelTrainer:
    def __init__(self,
                 model_name, 
                 model, 
                 num_epochs,
                 train_loader, 
                 val_loader,
                #  test_loader,
                 pos_enc_dim, 
                 optimizer, 
                 loss_fn_name,
                 loss_weights,
                 is_save_model,
                 save_model_dir, 
                 logger):
        
        self.model_name = model_name
        self.model = model
        self.num_epochs = num_epochs

        self.train_loader = train_loader
        self.val_loader = val_loader
        # self.test_loader = test_loader
        self.pos_enc_dim = pos_enc_dim

        self.optimizer = optimizer
        self.loss_fn = loss_fn_name
        self.loss_fn = LossFunctions(loss_fn_name).loss_fn
        self.loss_weights = loss_weights
        
        self.is_save_model = is_save_model
        self.save_model_dir = save_model_dir
        
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)

        self.logger = logger
        self.logger.log({'loss_weights': wandb.Table(data=[self.loss_weights], 
                                                     columns=['w1', 'w2', 'w3'])})

    def train_sub_step(self, inputs, targets1, targets2):
        inputs = positional_encoding(inputs, self.pos_enc_dim)

        outputs = self.model(inputs, targets1.grid, targets2.grid)
        return outputs

    def train(self):
        min_val_loss = float('inf')
        for epoch in range(self.num_epochs):
            self.model.train()
            total_loss = 0
            avg_loss_1 = 0
            avg_loss_2 = 0
            avg_loss_3 = 0
            for batch in tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{self.num_epochs}'):
                obj_names, vdb_32s, vdb_64s, vdb_128s = batch
                vdb_32s = fvdb.jcat(vdb_32s)
                vdb_64s = fvdb.jcat(vdb_64s)
                vdb_128s = fvdb.jcat(vdb_128s)
                self.optimizer.zero_grad()  

                # output_64 = self.train_sub_step(vdb_32s, vdb_64s)
                # output_128 = self.train_sub_step(vdb_64s, vdb_128s)
                output_128 = self.train_sub_step(vdb_32s, vdb_64s, vdb_128s)

                # Compute losses for each output and target
                # loss_1 = self.loss_fn(output_64.jdata, vdb_64s.jdata)
                loss_2 = self.loss_fn(output_128.jdata, vdb_128s.jdata)
                # loss_3 = self.loss_fn(output_64_128.jdata, vdb_128s.jdata) 

                # Combine losses (sum or weighted sum)
                [w1, w2, w3] = self.loss_weights
                loss = loss_2

                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()
                # avg_loss_1 += loss_1.item()
                # avg_loss_2 += loss_2.item()
                # avg_loss_3 += loss_3.item()
            avg_loss = total_loss / len(self.train_loader)
            # avg_loss_1 /= len(self.train_loader)
            # avg_loss_2 /= len(self.train_loader)
            # avg_loss_3 /= len(self.train_loader)
            print(f"Epoch {epoch+1}/{self.num_epochs}, Loss: {avg_loss:.4f}")
            if self.val_loader:
                (avg_val_loss) = self.validation()
            
            # Log the training loss
            self.logger.log({
                'train_loss': avg_loss,
                # 'train_loss_32->64': avg_loss_1,
                # 'train_loss_64->128': avg_loss_2,
                # 'train_loss_32->64->128': avg_loss_3,
                'val_loss': avg_val_loss,
                # 'val_loss_32->64': avg_val_loss_1,
                # 'val_loss_64->128': avg_val_loss_2,
                # 'val_loss_32->64->128': avg_val_loss_3,
                'epoch': epoch + 1
            })
            
            # Check if validation loss is lower than the minimum recorded loss
            # if avg_val_loss < min_val_loss:
            #     min_val_loss = avg_val_loss
            #     if self.is_save_model:
            #         self.save_model()
        
        print(f"Training complete. Minimum validation loss: {min_val_loss:.4f}")
        
    def validation(self):
        self.model.eval()
        total_loss = 0
        avg_loss_1 = 0
        avg_loss_2 = 0
        avg_loss_3 = 0
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validation'):
                obj_names, vdb_32s, vdb_64s, vdb_128s = batch
                vdb_32s = fvdb.jcat(vdb_32s)
                vdb_64s = fvdb.jcat(vdb_64s)
                vdb_128s = fvdb.jcat(vdb_128s)  

                # output_64 = self.train_sub_step(vdb_32s, vdb_64s)
                # output_128 = self.train_sub_step(vdb_64s, vdb_128s)
                output_128 = self.train_sub_step(vdb_32s, vdb_64s, vdb_128s)

                # loss1 = self.loss_fn(output_64.jdata, vdb_64s.jdata)
                loss2 = self.loss_fn(output_128.jdata, vdb_128s.jdata)
                # loss3 = self.loss_fn(output_64_128.jdata, vdb_128s.jdata)

                [w1, w2, w3] = self.loss_weights
                loss =  loss2

                total_loss += loss.item()
                # avg_loss_1 += loss1.item()
                # avg_loss_2 += loss2.item()
                # avg_loss_3 += loss3.item()
                
        avg_loss = total_loss / len(self.val_loader)
        # avg_loss_1 /= len(self.val_loader)
        # avg_loss_2 /= len(self.val_loader)
        # avg_loss_3 /= len(self.val_loader)
        print(f"Validation Loss: {avg_loss:.4f}")
        return avg_loss

    def save_model(self):
        path = os.path.join(self.save_model_dir, f"{self.model_name}.pth")
        torch.save(self.model, path)
        print(f"Model saved to {path}")

In [71]:
trainer = ModelTrainer(
                        model_name=config['training']['model_name'],
                        model=model,
                        num_epochs=10,
                        train_loader=train_dataloader,
                        val_loader=val_dataloader,
                        pos_enc_dim=4,
                        optimizer=optimizer,
                        loss_fn_name=config['training']['loss_function'],
                        loss_weights=config['training']['loss_weights'],
                        is_save_model=config['training']['save_model'],
                        save_model_dir=config['training']['save_model_dir'],
                        logger=logger
                    )

In [72]:
trainer.train()

Epoch 1/10: 100%|██████████| 201/201 [01:15<00:00,  2.65it/s]


Epoch 1/10, Loss: 0.2111


Validation: 100%|██████████| 67/67 [00:57<00:00,  1.16it/s]


Validation Loss: 0.1031


Epoch 2/10: 100%|██████████| 201/201 [01:18<00:00,  2.55it/s]


...

Validation Loss: 0.0937


Epoch 10/10: 100%|██████████| 201/201 [01:16<00:00,  2.63it/s]


Epoch 10/10, Loss: 0.1776


Validation: 100%|██████████| 67/67 [00:57<00:00,  1.17it/s]

Validation Loss: 0.0957
Training complete. Minimum validation loss: inf


In [ ]:

# evaluator = ABC_eval.Evaluator(
#                         model_name=config['training']['model_name'],
#                         pos_enc_dim=config['training']['positional_encoding'],
#                         test_loader=test_dataloader,
#                         upsampling_level=config['eval']['upsampling_level'],
#                         abc_dir=config['eval']['eval_dir'],
#                         save_model_dir=config['training']['save_model_dir'],
#                         save_predictions_dir=config['eval']['save_predictions_dir'],
#                         n_job=config['eval']['eval_job'],
#                         logger=logger
#                     )
# evaluator.evaluate()